# Ratio-метрики и пользовательские функции метрик в Ambrosia

В A/B-тестах часто ключевые метрики — это отношения: ARPU (revenue / users), средний чек (revenue / orders), CTR (clicks / impressions).

**Проблема:** наивный подход — посчитать ratio для каждого пользователя и применить t-test — даёт смещённую оценку дисперсии и некорректные p-value.

**Решение:** линеаризация ratio-метрики по формуле:

$$\text{linearized}_i = \text{numerator}_i - \hat{\theta} \cdot \text{denominator}_i$$

где $\hat{\theta} = \frac{\overline{\text{numerator}}}{\overline{\text{denominator}}}$ оценивается на контрольной группе.

Линеаризованная метрика приближённо нормальна, что позволяет корректно применять t-test.

В этом ноутбуке:
1. `LinearizationTransformer` — линеаризация ratio-метрик
2. `Preprocessor.linearize()` — интеграция в pipeline
3. `metric_funcs` — передача произвольных функций в `Tester`

## 1. Подготовка данных

Сгенерируем синтетические данные e-commerce эксперимента: пользователи совершают заказы с определённой выручкой. Группа B получила новую рекомендательную систему, которая увеличила средний чек.

In [ ]:
import numpy as np
import pandas as pd

from ambrosia.preprocessing import LinearizationTransformer, Preprocessor
from ambrosia.tester import Tester

np.random.seed(42)

n_users = 2000

# Контрольная группа A: orders ~ Poisson(5), revenue ~ orders * Normal(50, 15)
# Тестовая группа B: средний чек на 8% выше
orders_a = np.random.poisson(5, n_users) + 1
revenue_a = orders_a * np.random.normal(50, 15, n_users)

orders_b = np.random.poisson(5, n_users) + 1
revenue_b = orders_b * np.random.normal(54, 15, n_users)  # +8% к среднему чеку

data = pd.DataFrame({
    "user_id": range(2 * n_users),
    "group": ["A"] * n_users + ["B"] * n_users,
    "revenue": np.concatenate([revenue_a, revenue_b]),
    "orders": np.concatenate([orders_a, orders_b]),
})

print(f"Размер: {len(data)} пользователей")
print(f"\nСредний чек (revenue/orders):")
print(f"  A: {data[data['group']=='A']['revenue'].sum() / data[data['group']=='A']['orders'].sum():.2f}")
print(f"  B: {data[data['group']=='B']['revenue'].sum() / data[data['group']=='B']['orders'].sum():.2f}")
data.head()

Размер: 4000 пользователей

Средний чек (revenue/orders):
  A: 49.66
  B: 53.99


,user_id,group,revenue,orders
0,0,A,451.798622,6
1,1,A,198.239437,5
2,2,A,308.071829,5
3,3,A,363.804281,6
4,4,A,62.329293,6


## 2. LinearizationTransformer

Линеаризуем ratio-метрику `revenue / orders`. Трансформер оценивает `ratio` на референсных данных (контрольная группа), затем применяет формулу:

$$\text{linearized}_i = \text{revenue}_i - \hat{\theta} \cdot \text{orders}_i$$

In [2]:
# Fit на контрольной группе
control = data[data["group"] == "A"].copy()

transformer = LinearizationTransformer()
transformer.fit(control, numerator="revenue", denominator="orders", transformed_name="avg_check_lin")

print(f"Оценённый ratio (средний чек контроля): {transformer.ratio:.2f}")

Оценённый ratio (средний чек контроля): 49.66


In [3]:
# Transform обеих групп (ratio зафиксирован по контролю)
data_lin = transformer.transform(data)
data_lin[["user_id", "group", "revenue", "orders", "avg_check_lin"]].head(10)

,user_id,group,revenue,orders,avg_check_lin
0,0,A,451.798622,6,153.859825
1,1,A,198.239437,5,-50.042894
2,2,A,308.071829,5,59.789498
3,3,A,363.804281,6,65.865483
4,4,A,62.329293,6,-235.609504
5,5,A,150.008336,4,-48.617529
6,6,A,184.171749,6,-113.767048
7,7,A,283.591964,5,35.309633
8,8,A,385.471721,7,37.876458
9,9,A,442.951324,8,45.699595


### Сравнение: наивный ratio vs линеаризация

Наивный подход — посчитать `revenue / orders` для каждого пользователя — даёт завышенную дисперсию и менее мощный тест.

In [4]:
# Наивный ratio
data_lin["naive_ratio"] = data_lin["revenue"] / data_lin["orders"]

# Тест на наивном ratio
tester_naive = Tester(dataframe=data_lin, column_groups="group", metrics="naive_ratio")
result_naive = tester_naive.run(method="theory", as_table=True)

# Тест на линеаризованной метрике
tester_lin = Tester(dataframe=data_lin, column_groups="group", metrics="avg_check_lin")
result_lin = tester_lin.run(method="theory", as_table=True)

print("Наивный ratio (revenue/orders per user):")
print(result_naive[["pvalue", "effect", "confidence_interval", "metric name"]].to_string(index=False))
print(f"\nЛинеаризованная метрика:")
print(result_lin[["pvalue", "effect", "confidence_interval", "metric name"]].to_string(index=False))

Наивный ratio (revenue/orders per user):
      pvalue   effect confidence_interval metric name
1.080184e-19 4.296898    (3.3742, 5.2196) naive_ratio

Линеаризованная метрика:
      pvalue    effect confidence_interval   metric name
2.434053e-18 26.646817  (20.6801, 32.6135) avg_check_lin


## 3. Preprocessor.linearize() — интеграция в pipeline

Линеаризацию можно использовать в chain-архитектуре `Preprocessor`, комбинируя с другими трансформациями. Параметры пайплайна можно сохранить и воспроизвести.

In [5]:
# Chain: удаление выбросов по revenue -> линеаризация
preprocessor = (
    Preprocessor(control.copy(), verbose=False)
    .robust("revenue", alpha=0.01)
    .linearize("revenue", "orders", transformed_name="avg_check_lin")
)

result = preprocessor.data()
print("Колонки после пайплайна:", list(result.columns))
print(f"Трансформации: {[str(t) for t in preprocessor.transformations()]}")
result.head()

Колонки после пайплайна: ['user_id', 'group', 'revenue', 'orders', 'avg_check_lin']
Трансформации: ['Robust preprocessing', 'Linearization transformation']


,user_id,group,revenue,orders,avg_check_lin
0,0,A,451.798622,6,154.768550
1,1,A,198.239437,5,-49.285623
2,2,A,308.071829,5,60.546769
3,3,A,363.804281,6,66.774208
4,4,A,62.329293,6,-234.700779


In [6]:
# Сериализация: сохранить параметры пайплайна и воспроизвести на новых данных
import os

store_path = "linearize_pipeline.json"
preprocessor.store_transformations(store_path)

# Загрузка и применение на тестовой группе
treatment = data[data["group"] == "B"].copy()
loaded = Preprocessor(treatment, verbose=False)
loaded.load_transformations(store_path)
loaded_result = loaded.apply_transformations()

os.remove(store_path)

print("Параметры воспроизведены на тестовой группе:")
loaded_result[["user_id", "revenue", "orders", "avg_check_lin"]].head()

Параметры воспроизведены на тестовой группе:


,user_id,revenue,orders,avg_check_lin
2000,2000,399.637303,8,3.597207
2001,2001,147.154057,5,-100.371003
2002,2002,549.670106,9,104.124997
2003,2003,273.508238,6,-23.521834
2004,2004,485.630093,7,139.095009


## 4. metric_funcs — пользовательские функции метрик

Иногда удобнее не создавать колонку заранее, а передать функцию вычисления метрики прямо в `Tester`. Параметр `metric_funcs` принимает словарь `{имя_метрики: callable}`, где callable получает `pd.DataFrame` группы и возвращает массив значений.

In [7]:
# Определяем функции метрик
def avg_check(df):
    """Средний чек = revenue / orders."""
    return (df["revenue"] / df["orders"]).values

def revenue_per_user(df):
    """Revenue per user (просто revenue)."""
    return df["revenue"].values

# Передаём в Tester — колонки "avg_check" нет в данных, но Tester использует функцию
tester = Tester(
    dataframe=data,
    column_groups="group",
    metrics=["avg_check", "revenue_per_user"],
    metric_funcs={
        "avg_check": avg_check,
        "revenue_per_user": revenue_per_user,
    },
)

tester.run(method="theory")

,first_type_error,pvalue,effect,confidence_interval,metric name,group A label,group B label
0,0.05,2.160367e-19,4.296898,"(3.2417, 5.3521)",avg_check,A,B
1,0.05,5.428330e-14,36.305000,"(25.5614, 47.0486)",revenue_per_user,A,B


### metric_funcs в run() — переопределение на лету

Функции из `run()` имеют приоритет над заданными в конструкторе. Это удобно для быстрых экспериментов с разными определениями метрики.

In [8]:
# В конструкторе задана одна функция, в run() — другая
tester = Tester(
    dataframe=data,
    column_groups="group",
    metric_funcs={"my_metric": lambda df: df["revenue"].values},
)

# run() переопределяет: теперь "my_metric" считает revenue / orders
result = tester.run(
    metrics=["my_metric"],
    metric_funcs={"my_metric": lambda df: (df["revenue"] / df["orders"]).values},
    method="theory",
)
result

,first_type_error,pvalue,effect,confidence_interval,metric name,group A label,group B label
0,0.05,1.080184e-19,4.296898,"(3.3742, 5.2196)",my_metric,A,B


### metric_funcs + bootstrap

Пользовательские функции работают и с эмпирическим (bootstrap) методом.

In [9]:
tester = Tester(
    dataframe=data,
    column_groups="group",
    metrics=["avg_check"],
    metric_funcs={"avg_check": avg_check},
)

tester.run(method="empiric")

,first_type_error,pvalue,effect,confidence_interval,metric name,group A label,group B label
0,0.05,3.552714e-15,4.296898,"(3.3787, 5.2187)",avg_check,A,B


## Заключение

| Инструмент | Когда использовать |
|---|---|
| `LinearizationTransformer` | Нужна корректная статистика для ratio-метрик (ARPU, средний чек, CTR). Fit на контроле, transform на всех группах |
| `Preprocessor.linearize()` | Линеаризация как часть пайплайна предобработки с сериализацией |
| `metric_funcs` | Быстрое тестирование производных метрик без создания колонок. Удобно для ad-hoc анализа |

Подробнее:
- Документация `Tester`: параметр `metric_funcs`
- Документация `Preprocessor`: метод `linearize()`
- Документация `LinearizationTransformer`: `fit()`, `transform()`, `fit_transform()`